# FTS Parameter Sweep & Out-of-Sample Backtest Evaluation

This notebook evaluates hyperparameter and architectural variations under **controlled isolation** (*ceteris paribus*) for any selected parameter sweep specification (`SweepSpec`).

### Core Principles:
1. **Fixed Baseline Hyperparameters:** All non-swept hyperparameters (learning rate, batch size, dropout, execution fees, slippage) are held strictly constant.
2. **Out-of-Sample Evaluation:** Models are trained on the training split, registered as candidate ONNX models in `ModelRegistryLog`, and backtested on an independent holdout test split using the `BacktestEngine`.
3. **Generalized Sweep Results & Visualizations:** Results are encapsulated in `SweepResult`, saved to JSON summary files, and visualized using interactive Plotly charts (`SweepVisualizer`) and standalone HTML reports (`HTMLSweepExporter`).

### 1. Import Dependencies & Set Pathing

In [ ]:
import os
import sys
import logging

# Ensure src and project modules are on path
sys.path.insert(0, os.path.abspath("../src"))

from trading_bot.config import settings
from trading_bot.backtesting import SweepResult, SweepVisualizer, HTMLSweepExporter
from plugins.nets.spec import SweepSpec
from plugins.nets.training.sweep_runner import run_parameter_sweep

# Set logging level
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

### 2. Load Sweep Specification & Execute Controlled Sweep

Set `spec_path` to point to any parameter sweep specification YAML file. The sweep runner dynamically inspects `spec.sweep_param` and `spec.sweep_values`.

In [ ]:
spec_path = "../specs/train/BTCUSDT/sweep/lstm_num_layers.yaml"
spec = SweepSpec.from_yaml(spec_path)

print(f"Loaded Sweep Spec     : '{spec.sweep_name}'")
print(f"Model Architecture    : {spec.model_type.upper()}")
print(f"Target Market         : {spec.market.market_id} ({spec.market.interval})")
print(f"Sweeping Parameter    : '{spec.sweep_param}' over {spec.sweep_values}")
print(f"Train Date Range      : {spec.train_dates.start_date} to {spec.train_dates.end_date}")
print(f"Test Date Range       : {spec.test_dates.start_date} to {spec.test_dates.end_date}")

sweep_result: SweepResult = run_parameter_sweep(spec)

print(f"\n=================== OUT-OF-SAMPLE SWEEP RESULTS ({sweep_result.sweep_param}) ===================")
print(f"Total Trials Completed: {len(sweep_result.trials)}")
print(f"Created At: {sweep_result.created_at}")

### 3. Interactive Sensitivity & Performance Visualization

Render the multi-view interactive Plotly chart combining parameter sensitivity curves, drawdown profiles, overlaid equity curves, and metric summary table.

In [ ]:
visualizer = SweepVisualizer()
fig = visualizer.render_charts(sweep_result)
fig.show()

### 4. Export Visualization Report

Export standalone interactive HTML report using `HTMLSweepExporter`.

In [ ]:
exporter = HTMLSweepExporter()
report_path = exporter.export(sweep_result)
print(f"Interactive Sweep Visualization HTML saved to: {report_path}")